In [3]:
import sys

!{sys.executable} -m pip install scikit-learn

'c:\Users\USER\OneDrive\Desktop\leakmind' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

print("scikit-learn imported successfully")

scikit-learn imported successfully


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

features = pd.read_csv("../data/processed/leakmind_features.csv")

print("Dataset shape:", features.shape)
features.head()

Dataset shape: (1000, 13)


,user,http_activity_count,after_hours_activity,total_logon_events,unique_pcs,logon_count,logoff_count,after_hours_logon,total_device_events,device_pcs,device_connect_count,device_disconnect_count,after_hours_device
0,DTAA/AAA0371,7085,0,843,16,536,307,0.0,0.0,0.0,0.0,0.0,0.0
1,DTAA/AAC0344,14884,0,690,1,345,345,0.0,0.0,0.0,0.0,0.0,0.0
2,DTAA/AAC0599,14926,0,690,1,345,345,0.0,0.0,0.0,0.0,0.0,0.0
3,DTAA/AAH0734,16740,0,690,1,345,345,0.0,0.0,0.0,0.0,0.0,0.0
4,DTAA/AAK0658,14715,0,690,1,345,345,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
user_ids = features["user"]

X = features.drop(columns=["user"])

print("ML feature shape:", X.shape)
print("Features:")
print(X.columns.tolist())

ML feature shape: (1000, 12)
Features:
['http_activity_count', 'after_hours_activity', 'total_logon_events', 'unique_pcs', 'logon_count', 'logoff_count', 'after_hours_logon', 'total_device_events', 'device_pcs', 'device_connect_count', 'device_disconnect_count', 'after_hours_device']


In [5]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Scaled feature shape:", X_scaled.shape)
print("Mean of first 3 features:", X_scaled[:, :3].mean(axis=0))
print("Std of first 3 features:", X_scaled[:, :3].std(axis=0))

Scaled feature shape: (1000, 12)
Mean of first 3 features: [-2.84217094e-17  0.00000000e+00  1.35003120e-16]
Std of first 3 features: [1. 1. 1.]


In [6]:
isolation_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

isolation_forest.fit(X_scaled)

print("Isolation Forest training completed.")

Isolation Forest training completed.


In [7]:
features["anomaly_score"] = isolation_forest.decision_function(X_scaled)

features["anomaly_flag"] = isolation_forest.predict(X_scaled)

print("Anomaly detection completed.")
print(features[["user", "anomaly_score", "anomaly_flag"]].head())

Anomaly detection completed.
           user  anomaly_score  anomaly_flag
0  DTAA/AAA0371       0.179333             1
1  DTAA/AAC0344       0.214876             1
2  DTAA/AAC0599       0.213669             1
3  DTAA/AAH0734       0.196935             1
4  DTAA/AAK0658       0.214753             1


In [8]:
print("Normal users:", (features["anomaly_flag"] == 1).sum())
print("Anomalous users:", (features["anomaly_flag"] == -1).sum())

Normal users: 950
Anomalous users: 50


In [9]:
top_anomalies = features.sort_values(
    by="anomaly_score",
    ascending=True
)

top_anomalies[
    ["user", "anomaly_score", "anomaly_flag"]
].head(20)

,user,anomaly_score,anomaly_flag
138,DTAA/BJM0992,-0.182457,-1
203,DTAA/CGM0994,-0.181428,-1
90,DTAA/ARS0993,-0.181198,-1
444,DTAA/IRC0991,-0.171768,-1
459,DTAA/JCC0998,-0.170578,-1
326,DTAA/ELD1000,-0.167417,-1
520,DTAA/KEE0997,-0.167290,-1
294,DTAA/DSM0990,-0.166861,-1
504,DTAA/JTT0989,-0.160083,-1
217,DTAA/CLN0999,-0.157339,-1


In [12]:
import os

os.makedirs("../data/processed", exist_ok=True)

anomaly_results = features[
    ["user", "anomaly_score", "anomaly_flag"]
]

anomaly_results.to_csv(
    "../data/processed/isolation_forest_results.csv",
    index=False
)

print("Isolation Forest results saved successfully.")
print("Shape:", anomaly_results.shape)

Isolation Forest results saved successfully.
Shape: (1000, 3)


In [13]:
leakmind_features = features.copy()

print("Final feature dataset shape:", leakmind_features.shape)
print(leakmind_features.head())

Final feature dataset shape: (1000, 15)
           user  http_activity_count  after_hours_activity  \
0  DTAA/AAA0371                 7085                     0   
1  DTAA/AAC0344                14884                     0   
2  DTAA/AAC0599                14926                     0   
3  DTAA/AAH0734                16740                     0   
4  DTAA/AAK0658                14715                     0   

   total_logon_events  unique_pcs  logon_count  logoff_count  \
0                 843          16          536           307   
1                 690           1          345           345   
2                 690           1          345           345   
3                 690           1          345           345   
4                 690           1          345           345   

   after_hours_logon  total_device_events  device_pcs  device_connect_count  \
0                0.0                  0.0         0.0                   0.0   
1                0.0                  0.0   

In [14]:
leakmind_features.to_csv(
    "../data/processed/leakmind_features_with_anomaly.csv",
    index=False
)

print("Combined LeakMind feature dataset saved.")
print("Shape:", leakmind_features.shape)

Combined LeakMind feature dataset saved.
Shape: (1000, 15)
